In [1]:
!pip install deepagents langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.9/341.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.1/813.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 22.1 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found ex

In [2]:
import asyncio
from deepagents import create_deep_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [3]:
api_key=""         # <-- paste your OpenAI key
MODEL = "gpt-5"

In [4]:
KNOWLEDGE_BASE = {
    "deep agents": "Deep agents combine a planning tool, a virtual file system, and subagents to tackle long-horizon tasks.",
    "subagents": "A subagent is a focused agent the orchestrator delegates to via the built-in `task` tool; it runs in its own context window.",
    "langgraph": "LangGraph models stateful, multi-actor LLM apps as graphs of nodes and edges with shared state.",
    "deploy policy": "Production deploys need sign-off from two on-call engineers first.",
}

In [5]:
@tool
def kb_lookup(topic: str) -> str:
    """Look up `topic` in the local knowledge base and return a fact."""
    topic_l = topic.lower()
    for key, fact in KNOWLEDGE_BASE.items():
        if key in topic_l or topic_l in key:
            return fact
    return f"No knowledge-base entry for {topic!r}."

@tool
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic `expression`, e.g. '12 * (3 + 4)'."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: only numbers and + - * / ( ) are allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Error: {exc}"




In [6]:



research_subagent = {
    "name": "research-agent",
    "description": "Use this to answer a factual question from the local knowledge base. Give it ONE focused question.",
    "system_prompt": (
        "You are a knowledge-base lookup agent. You have NO knowledge of your own. "
        "For EVERY question you MUST call the kb_lookup tool first, and answer using "
        "ONLY the exact text it returns. Quote that text verbatim. If kb_lookup returns "
        "a 'No knowledge-base entry' message, reply that the knowledge base has no entry "
        "for it. Never answer from your own general knowledge, and do not ask follow-ups."
    ),
    "tools": [kb_lookup],
}

math_subagent = {
    "name": "math-agent",
    "description": "Use this for any arithmetic or numeric calculation.",
    "system_prompt": "You are a careful calculator. Use the calculator tool for every computation. Report only the result and the expression you evaluated.",
    "tools": [calculator],
}

ORCHESTRATOR_INSTRUCTIONS = """You are a research orchestrator. You have NO knowledge of your own and must never answer from general knowledge. Delegate every part of the request via the `task` tool:
  - any factual lookup  -> call task with subagent_type="research-agent"
  - any arithmetic      -> call task with subagent_type="math-agent"
When you delegate, pass the relevant question to sub-agent. Do NOT invent extra sub-questions, examples, or requirements in the task description.
Wait for the subagents' results, then build ONE final answer using ONLY the text those subagents returned. Do not add, infer, or embellish beyond their results. If a subagent reports no knowledge-base entry, say so plainly. Keep it tight."""

agent = create_deep_agent(
    tools=[kb_lookup, calculator],
    system_prompt=ORCHESTRATOR_INSTRUCTIONS,
    subagents=[research_subagent, math_subagent],
    model=ChatOpenAI(model=MODEL, api_key=api_key),
)


In [7]:
import nest_asyncio
nest_asyncio.apply()

In [8]:
result = agent.invoke({
    "messages": [{"role": "user", "content":
        "what are subagents , and also compute 12 * (3 + 4)."}]
})


for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

what are subagents , and also compute 12 * (3 + 4).
================================== Ai Message ==================================
Tool Calls:
  task (call_870QxDFZadveAsdh7ZZrWcmi)
 Call ID: call_870QxDFZadveAsdh7ZZrWcmi
  Args:
    description: Answer the question: What are subagents? Return a concise definition.
    subagent_type: research-agent
  task (call_QrABdOgcEqrpdH69SHJhW3Ur)
 Call ID: call_QrABdOgcEqrpdH69SHJhW3Ur
  Args:
    description: Compute 12 * (3 + 4). Return only the numeric result.
    subagent_type: math-agent
================================= Tool Message =================================
Name: task

"A subagent is a focused agent the orchestrator delegates to via the built-in `task` tool; it runs in its own context window."
================================= Tool Message =================================
Name: task

84
================================== Ai Message ===============